In [1]:
import numpy as np
import pandas as pd

# ==========================================
# INTERACTIVE INPUTS — UPDATED FOR BACKTESTING
# ==========================================
print("=== TIERED PRICE-INDEX INSURANCE — BOOTSTRAP MONTE CARLO ===")
print("Enter the values for this pricing date (e.g. January 2023)\n")

S0_rss1 = float(input("Enter RSS1 spot price S0 (LKR) [e.g. 760.00]: ") or 760.00)
S0_rss3 = float(input("Enter RSS3 spot price S0 (LKR) [e.g. 686.25]: ") or 686.25)
prod_cost = float(input("Enter Production cost (LKR) [e.g. 386.08]: ") or 386.08)
r = float(input("Enter Risk-free rate (as decimal, e.g. 0.0890 for 8.90%): ") or 0.0890)

T = 1.0          # 1-year contract (fixed)
SIMS = 10000     # Number of simulations (fixed)
SEED = 42        # Random seed for reproducibility (fixed)

# Historical means for strike tiers (fixed - not latest prices)
hist_mean_rss1 = 320.25
hist_mean_rss3 = 307.88

print("\n✅ Inputs accepted!")
print(f"   RSS1 S0 = {S0_rss1:.2f} | RSS3 S0 = {S0_rss3:.2f}")
print(f"   Production Cost = {prod_cost:.2f} | Risk-free rate = {r*100:.2f}%\n")

# ==========================================
# BOOTSTRAP MONTE CARLO FUNCTION (PRICE ONLY)
# ==========================================
def bootstrap_mc_premium(S0, K, r, T, historical_log_returns, simulations=SIMS, seed=SEED):
    """Pure price-index insurance premium using Bootstrap Monte Carlo."""
    months_to_simulate = int(round(12 * T))
    rng = np.random.default_rng(seed)
    draws = rng.choice(historical_log_returns,
                       size=(simulations, months_to_simulate),
                       replace=True)
    annual_log_return = draws.sum(axis=1)
    ST = S0 * np.exp(annual_log_return)
    payouts = np.maximum(K - ST, 0)
    premium = np.exp(-r * T) * payouts.mean()
    return premium

# ==========================================
# LOAD REAL DATA FROM CSV (FOR LOG RETURNS ONLY)
# ==========================================
df = pd.read_csv("Data_Python.csv")
df = df.rename(columns={
    "YEAR": "Date",
    "Final_Price_RSS 1": "RSS1",
    "Final_Price_RSS 3": "RSS3"
})
df["RSS1"] = pd.to_numeric(df["RSS1"], errors="coerce")
df["RSS3"] = pd.to_numeric(df["RSS3"], errors="coerce")
df = df.dropna(subset=["RSS1", "RSS3"]).reset_index(drop=True)
df["Date"] = pd.to_datetime(df["Date"].str.title(), format="%Y%b", errors="coerce")
df = df.dropna(subset=["Date"]).sort_values("Date").reset_index(drop=True)
df = df[(df["RSS1"] > 0) & (df["RSS3"] > 0)].reset_index(drop=True)

df["rss1_log_return"] = np.log(df["RSS1"] / df["RSS1"].shift(1))
df["rss3_log_return"] = np.log(df["RSS3"] / df["RSS3"].shift(1))

rss1_historical_returns = df["rss1_log_return"].dropna().to_numpy()
rss3_historical_returns = df["rss3_log_return"].dropna().to_numpy()

# ==========================================
# RUN MCS FOR ALL SCENARIOS (PRICE ONLY)
# ==========================================
grades = ["RSS 1", "RSS 3"]
s0_map = {"RSS 1": S0_rss1, "RSS 3": S0_rss3}
hmean_map = {"RSS 1": hist_mean_rss1, "RSS 3": hist_mean_rss3}
returns_map = {"RSS 1": rss1_historical_returns, "RSS 3": rss3_historical_returns}

results = []
for grade in grades:
    S = s0_map[grade]
    h_mean = hmean_map[grade]
    scenarios = {
        '100% Market Price' : S,
        '95% Market Price'  : S * 0.95,
        'Historical Mean'   : h_mean,
        'Production Cost'   : prod_cost,
    }
    for label, K in scenarios.items():
        premium_base = bootstrap_mc_premium(
            S, K, r, T, returns_map[grade]
        )
        results.append({
            'Grade' : grade,
            'Scenario' : label,
            'Strike K (LKR)' : round(K, 2),
            'MCS Base Premium (LKR)': round(premium_base, 4)
        })

# ==========================================
# PRINT RESULTS
# ==========================================
df_final = pd.DataFrame(results)
print("\n" + "=" * 85)
print(" TIERED PRICE-INDEX INSURANCE — BOOTSTRAP MONTE CARLO (PRICE ONLY)")
print("=" * 85)
print(f" RSS1 S0 : {S0_rss1:.2f} LKR | RSS3 S0 : {S0_rss3:.2f} LKR")
print(f" RSS1 Historical Mean : {hist_mean_rss1:.2f} LKR | RSS3 Historical Mean : {hist_mean_rss3:.2f} LKR")
print(f" Production Cost : {prod_cost:.2f} LKR")
print(f" Risk-free rate : {r*100:.2f}% | T : {T} year")
print(f" Simulations : {SIMS:,} | Random seed : {SEED}")
print("=" * 85)

for grade in grades:
    print(f" -- {grade} --")
    df_g = df_final[df_final['Grade'] == grade][
        ['Scenario', 'Strike K (LKR)', 'MCS Base Premium (LKR)']
    ]
    print(df_g.to_string(index=False))
    print()

print("=" * 85)
print(" NOTE: These are pure price-protection premiums (no weather adjustment)")
print("=" * 85)

=== TIERED PRICE-INDEX INSURANCE — BOOTSTRAP MONTE CARLO ===
Enter the values for this pricing date (e.g. January 2023)



Enter RSS1 spot price S0 (LKR) [e.g. 760.00]:  325.34
Enter RSS3 spot price S0 (LKR) [e.g. 686.25]:  308
Enter Production cost (LKR) [e.g. 386.08]:  210
Enter Risk-free rate (as decimal, e.g. 0.0890 for 8.90%):  0.0858



✅ Inputs accepted!
   RSS1 S0 = 325.34 | RSS3 S0 = 308.00
   Production Cost = 210.00 | Risk-free rate = 8.58%


 TIERED PRICE-INDEX INSURANCE — BOOTSTRAP MONTE CARLO (PRICE ONLY)
 RSS1 S0 : 325.34 LKR | RSS3 S0 : 308.00 LKR
 RSS1 Historical Mean : 320.25 LKR | RSS3 Historical Mean : 307.88 LKR
 Production Cost : 210.00 LKR
 Risk-free rate : 8.58% | T : 1.0 year
 Simulations : 10,000 | Random seed : 42
 -- RSS 1 --
         Scenario  Strike K (LKR)  MCS Base Premium (LKR)
100% Market Price          325.34                 24.0436
 95% Market Price          309.07                 17.4254
  Historical Mean          320.25                 21.8381
  Production Cost          210.00                  0.4909

 -- RSS 3 --
         Scenario  Strike K (LKR)  MCS Base Premium (LKR)
100% Market Price          308.00                 22.5014
 95% Market Price          292.60                 16.2052
  Historical Mean          307.88                 22.4479
  Production Cost          210.00           